In [5]:
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import pandas as pd
import json
import csv
import pickle
import numpy as np
        
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from elasticsearch import Elasticsearch

In [6]:
res = requests.get("http://localhost:9200")
es = Elasticsearch([{"host":"localhost", "port": 9200}])

counter = 0
index_name = "dc_cubes_historic"
predictionColumn = "cpuusage_ps"


In [8]:
def deleteElasticsearchIndex(index_name, deleteEverything=True):  # löscht immer den ganzen index
    if (deleteEverything):
        es.indices.delete(index_name)
    print(index_name, " index deleted")

In [9]:
def pushData(row):
    global core_name
    global counter
    global allMetrics
    global predictionColumn
    # defining the api-endpoint
    url = "http://localhost:8983/solr/"+core_name+"/update/json/docs"
    # data to be sent to api
    data = {
                "timestamp": str(row["timestamp"]),
                "cluster": row["cluster"],
                "dc": row["dc"],
                "perm": -1, #row["perm"],
                "instanz": row["instanz"],
                "verfahren": "-1",# row["verfahren"],
                "service": "-1" ,#row["verfahren"],
                "response": -1 ,#row["response"],
                "count": row[predictionColumn],
                "minv":  -1,#row["minv"],
                "maxv":  -1,#row["maxv"],
                "avg": -1, #row["avg"],
                "var": -1 ,#row["var"],
                "dev_upp": -1, #row["dev_upp"],
                "dev_low": -1, #row["dev_low"],
                "perc90": -1 ,#row["perc90"],
                "perc95": -1 ,#row["perc95"],
                "perc99": -1 ,#row["perc99.9"],
                "sum": -1, #row["sum"],
                "sum_of_squares": -1, #row["sum_of_squares"],
                "server": "PBZ0%dE00_PERM02_S0%s_OSB" % (row["cluster"], row["instanz"]), #row["server"]
            }
    
    for metric in allMetrics:
        data[metric] = row[metric]
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.get(url)
    headers = {'Content-type': 'application/json'}
    # sending post request
    session.post(url=url, data=json.dumps(data), headers=headers)
    counter += 1
    if (counter % 1000 == 0):
        print("Commiting... counter:", counter)
        requests.get("http://localhost:8983/solr/"+core_name+"/update?commit=true")

In [10]:
def createElasticsearchIndex(index_name):
    request_body = {
                "settings" : {
                    "number_of_shards": 5,
                    "number_of_replicas": 1
                },
                "mappings": {
                    "properties": {
                        "@timestamp": {
                            "type":"date"
                        },
                        "cluster": {
                            "type": "long"
                        },
                        "dc": {
                            "type": "long"
                        },
                        "perm": {
                            "type": "long"
                        },    
                        "instanz": {
                            "type": "long"
                        },
                        "verfahren": {
                            "type": "text"
                        },
                        "service": {
                            "type": "text"
                        },  
                        "response": {
                            "type": "long"
                        },  
                        "count": {
                            "type": "long"
                        },
                        "minv": {
                            "type": "long"
                        },
                        "maxv": {
                            "type": "long"
                        },
                        "avg": {
                            "type": "float"
                        },
                        "var": {
                            "type": "float"
                        },
                        "dev_upp": {
                            "type": "float"
                        },
                         "dev_low": {
                            "type": "float"
                        },
                        "perc90": {
                            "type": "float"
                        },
                        "perc95": {
                            "type": "float"
                        },
                        "perc99": {
                            "type": "float"
                        },
                        "sum": {
                            "type": "text"
                        },
                        "sum_of_squares": {
                            "type": "long"
                        },
                        "server": {
                            "type": "text"
                        },
                        "index": { 
                            "type": "float"
                        },
                        "transactions_ps": {
                            "type": "float"
                        },
                        "opt_sga_size": {
                            "type": "float"
                        },
                        "sga_total": {
                            "type": "float"
                        },
                        "fixed_sga": {
                            "type": "float"
                        },
                        "java_pool": {
                            "type": "float"
                        },
                        "shared_pool": {
                            "type": "float"
                        },
                        "buffer_cache": {
                            "type": "float"
                        },
                        "pga_total": {
                            "type": "float"
                        },
                        "other_sga_memory": {
                            "type": "float"
                        },
                        "large_pool": {
                            "type": "float"
                        },
                        "streams_pool": {
                            "type": "float"
                        },
                        "log_buffer": {
                            "type": "float"
                        },
                        "total_pga_allocated": {
                            "type": "float"
                        },
                        "shared_free_pct": {
                            "type": "float"
                        },
                        "large_free_pct": {
                            "type": "float"
                        },
                        "java_free_pct": {
                            "type": "float"
                        },
                        "dbTimePs": {
                            "type": "float"
                        },
                        "inboundPlsqlRpcElapsedTimePs": {
                            "type": "float"
                        },
                        "sqlExecuteElapsedTimePs": {
                            "type": "float"
                        },
                        "failedParseElapsedTimePs": {
                            "type": "float"
                        },
                        "javaExecutionElapsedTimePs": {
                            "type": "float"
                        },
                        "hardParseElapsedTimePs": {
                            "type": "float"
                        },
                        "parseTimeElapsedPs": {
                            "type": "float"
                        },
                        "failedParseSharMemElTimePs": {
                            "type": "float"
                        },
                        "hardParseBindMismElTimePs": {
                            "type": "float"
                        },
                        "bgCpuTimePs": {
                            "type": "float"
                        },
                        "hardParseSharCritElTimePs": {
                            "type": "float"
                        },
                        "plsqlCompilationElapsedTimePs": {
                            "type": "float"
                        },
                        "connManageCallElapsedTimePs": {
                            "type": "float"
                        },
                        "sequenceLoadElapsedTimePs": {
                            "type": "float"
                        },
                        "repeatedBindElapsedTimePs": {
                            "type": "float"
                        },
                        "dbCpuPs": {
                            "type": "float"
                        },
                        "plsqlEexecutionElapsedTimePs": {
                            "type": "float"
                        },
                        "rmanCpuTimeBackupRestorePs": {
                            "type": "float"
                        },
                        "total_memory": {
                            "type": "float"
                        },
                        "logons": {
                            "type": "float"
                        },
                        "process_usage": {
                            "type": "float"
                        },
                        "user_limit": {
                            "type": "float"
                        },
                        "opencursors": {
                            "type": "float"
                        },
                        "session_usage": {
                            "type": "float"
                        },
                        "averageWaitTimeFG": {
                            "type": "float"
                        },
                        "totalWaitsFG": {
                            "type": "float"
                        },
                        "averageWaitTime": {
                            "type": "float"
                        },
                        "totalWaits": {
                            "type": "float"
                        },
                        "totalWaitTimeFG": {
                            "type": "float"
                        },
                        "totalWaitTime": {
                            "type": "float"
                        },
                        "dbtime_waitclass_pct": {
                            "type": "float"
                        },
                        "user_wait_time_pct": {
                            "type": "float"
                        },
                        "avg_users_waiting_on_class": {
                            "type": "float"
                        },
                        "avg_user_cpu_time_pct": {
                            "type": "float"
                        },
                        "user_cpu_time_cnt": {
                            "type": "float"
                        },
                        "other_wait_cnt": {
                            "type": "float"
                        },
                        "userio_wait_cnt": {
                            "type": "float"
                        },
                        "host_cpu_usage_pct": {
                            "type": "float"
                        },
                        "opncurs_pt": {
                            "type": "float"
                        },
                        "failedparses_ps": {
                            "type": "float"
                        },
                        "parses_ps": {
                            "type": "float"
                        },
                        "physreads_pt": {
                            "type": "float"
                        },
                        "crundorecs_pt": {
                            "type": "float"
                        },
                        "branchnodesplits_pt": {
                            "type": "float"
                        },
                        "indxscansfull_pt": {
                            "type": "float"
                        },
                        "executeswoparse_pct": {
                            "type": "float"
                        },
                        "redowrites_pt": {
                            "type": "float"
                        },
                        "crundorecs_ps": {
                            "type": "float"
                        },
                        "executions_ps": {
                            "type": "float"
                        },
                        "consistentreadgets_ps": {
                            "type": "float"
                        },
                        "tabscanslong_ps": {
                            "type": "float"
                        },
                        "physwriteslob_pt": {
                            "type": "float"
                        },
                        "physwritesdir_ps": {
                            "type": "float"
                        },
                        "sortsmemory_ps": {
                            "type": "float"
                        },
                        "dbblkgets_pt": {
                            "type": "float"
                        },
                        "dbblkgets_ps": {
                            "type": "float"
                        },
                        "userrollbackundorec_ps": {
                            "type": "float"
                        },
                        "redosize_pt": {
                            "type": "float"
                        },
                        "indxscanstotal_ps": {
                            "type": "float"
                        },
                        "redosize_ps": {
                            "type": "float"
                        },
                        "leafnodesplits_ps": {
                            "type": "float"
                        },
                        "physreadslob_pt": {
                            "type": "float"
                        },
                        "bgcheckpoints_ps": {
                            "type": "float"
                        },
                        "leafnodesplits_pt": {
                            "type": "float"
                        },
                        "usercalls_ps": {
                            "type": "float"
                        },
                        "sortsmemory_pt": {
                            "type": "float"
                        },
                        "userrollbackundorec_pt": {
                            "type": "float"
                        },
                        "physreadsdir_pt": {
                            "type": "float"
                        },
                        "redowrites_ps": {
                            "type": "float"
                        },
                        "indxscansfull_ps": {
                            "type": "float"
                        },
                        "enqwaits_ps": {
                            "type": "float"
                        },
                        "branchnodesplits_ps": {
                            "type": "float"
                        },
                        "physwrites_ps": {
                            "type": "float"
                        },
                        "indxscanstotal_pt": {
                            "type": "float"
                        },
                        "enqreqs_pt": {
                            "type": "float"
                        },
                        "sortsrows_pt": {
                            "type": "float"
                        },
                        "rollbacks_pt": {
                            "type": "float"
                        },
                        "usercall_pct": {
                            "type": "float"
                        },
                        "recurscalls_ps": {
                            "type": "float"
                        },
                        "enqreqs_ps": {
                            "type": "float"
                        },
                        "sortsdisk_ps": {
                            "type": "float"
                        },
                        "executions_pt": {
                            "type": "float"
                        },
                        "dbtime_ps": {
                            "type": "float"
                        },
                        "enqwaits_pt": {
                            "type": "float"
                        },
                        "crblks_pt": {
                            "type": "float"
                        },
                        "physwritesdir_pt": {
                            "type": "float"
                        },
                        "physwrites_pt": {
                            "type": "float"
                        },
                        "enqtimeouts_ps": {
                            "type": "float"
                        },
                        "sortsrows_ps": {
                            "type": "float"
                        },
                        "iorequests_ps": {
                            "type": "float"
                        },
                        "physwriteslob_ps": {
                            "type": "float"
                        },
                        "logons_pt": {
                            "type": "float"
                        },
                        "enqdeadlocks_pt": {
                            "type": "float"
                        },
                        "consistentreadchanges_ps": {
                            "type": "float"
                        },
                        "hardparses_pt": {
                            "type": "float"
                        },
                        "sortsdisk_pt": {
                            "type": "float"
                        },
                        "tabscanstotal_ps": {
                            "type": "float"
                        },
                        "iombs_ps": {
                            "type": "float"
                        },
                        "avg_sync_singleblk_read_latency": {
                            "type": "float"
                        },
                        "tabscanslong_pt": {
                            "type": "float"
                        },
                        "failedparses_pt": {
                            "type": "float"
                        },
                        "commits_pt": {
                            "type": "float"
                        },
                        "consistentreadgets_pt": {
                            "type": "float"
                        },
                        "logons_ps": {
                            "type": "float"
                        },
                        "dbblkchanges_ps": {
                            "type": "float"
                        },
                        "opncurs_ps": {
                            "type": "float"
                        },
                        "logreads_ps": {
                            "type": "float"
                        },
                        "dbblkchanges_pt": {
                            "type": "float"
                        },
                        "dbwrcheckpoints_ps": {
                            "type": "float"
                        },
                        "enqdeadlocks_ps": {
                            "type": "float"
                        },
                        "networkbytes_ps": {
                            "type": "float"
                        },
                        "parses_pt": {
                            "type": "float"
                        },
                        "usercalls_pt": {
                            "type": "float"
                        },
                        "rollbacks_ps": {
                            "type": "float"
                        },
                        "physreadslob_ps": {
                            "type": "float"
                        },
                        "rows_psort": {
                            "type": "float"
                        },
                        "softparse_pct": {
                            "type": "float"
                        },
                        "physreadsdir_ps": {
                            "type": "float"
                        },
                        "enqtimeouts_pt": {
                            "type": "float"
                        },
                        "tabscanstotal_pt": {
                            "type": "float"
                        },
                        "crblks_ps": {
                            "type": "float"
                        },
                        "hardparses_ps": {
                            "type": "float"
                        },
                        "consistentreadchanges_pt": {
                            "type": "float"
                        },
                        "dbtime_pt": {
                            "type": "float"
                        },
                        "avg_active_sessions": {
                            "type": "float"
                        },
                        "recurscalls_pt": {
                            "type": "float"
                        },
                        "physreads_ps": {
                            "type": "float"
                        },
                        "logreads_pt": {
                            "type": "float"
                        },
                        "commits_ps": {
                            "type": "float"
                        },
                        "interconnect_rate": {
                            "type": "float"
                        },
                        "corrupt": {
                            "type": "float"
                        },
                        "lost": {
                            "type": "float"
                        },
                        "cr_request_cs": {
                            "type": "float"
                        },
                        "currentgets_cs": {
                            "type": "float"
                        },
                        "time_cs": {
                            "type": "float"
                        },
                        "pxdwngrdserial_pt": {
                            "type": "float"
                        },
                        "response_time_pt": {
                            "type": "float"
                        },
                        "cpu_time_pct": {
                            "type": "float"
                        },
                        "libcache_miss_pct": {
                            "type": "float"
                        },
                        "dictionaryhit_pct": {
                            "type": "float"
                        },
                        "pxdwngrd75_ps": {
                            "type": "float"
                        },
                        "pgacachehit_pct": {
                            "type": "float"
                        },
                        "libcache_hit_pct": {
                            "type": "float"
                        },
                        "inmem_sort_pct": {
                            "type": "float"
                        },
                        "pxdwngrdserial_ps": {
                            "type": "float"
                        },
                        "cursorcachehit_pct": {
                            "type": "float"
                        },
                        "bufcachehit_pct": {
                            "type": "float"
                        },
                        "redologalloc_hit_pct": {
                            "type": "float"
                        },
                        "cpuusage_pt": {
                            "type": "float"
                        },
                        "pxdwngrd25_ps": {
                            "type": "float"
                        },
                        "dictionarymiss_pct": {
                            "type": "float"
                        },
                        "cpuusage_ps": {
                            "type": "float"
                        },
                        "pxdwngrd50_ps": {
                            "type": "float"
                        },
                        "avg_tot_cpu_usage_ps": {
                            "type": "float"
                        },
                        "max_tot_cpu_usage_ps": {
                            "type": "float"
                        },
                        "dayOfWeek": {
                            "type": "float"
                        },
                        "isWeekend": {
                            "type": "float"
                        },
                        "hour": {
                            "type": "float"
                        },
                        "minute": {
                            "type": "float"
                        },
                        "id": {
                            "type": "text"
                        },
                        "_version_": {
                            "type": "long"
                        }
                    }
                },
            }
    
    es.indices.create(index = new_index_name, body = request_body)
    print(index_name, " created")


In [11]:
def initSchema(core_name, allMetrics):
    url = "http://localhost:8983/solr/"+core_name+"/schema"
    headers = {'Content-type': 'application/json'}
    rowsDict = {
        "timestamp": "pdate", "host": "string", "cluster": "pint", "dc": "pint", "perm": "pint", "instanz": "string", "verfahren": "string",
        "service": "string", "response": "pint", "count": "pfloat", "minv": "pint", "maxv": "pint", "avg": "pfloat", "var": "pfloat",
        "dev_upp": "pfloat", "dev_low": "pfloat", "perc90": "pfloat", "perc95": "pfloat", "perc99": "pfloat", "sum": "pint",
        "sum_of_squares": "pint", "server": "string"}

    for name in rowsDict:
        data = {
            "add-field": {"stored": "true", "docValues": "true", "indexed": "false", "multiValued": "false", "name": name, "type": rowsDict[name]}
        }
        requests.post(url=url, data=json.dumps(data), headers=headers)
        
    for metric in allMetrics:
        if metric not in rowsDict:
            data = {
                "add-field": {"stored": "true", "docValues": "true", "indexed": "false", "multiValued": "false", "name": metric, "type": "pfloat"}
            }
            requests.post(url=url, data=json.dumps(data), headers=headers)
    
    
    print(core_name, " schema inited")


In [12]:
def deleteCoreDocuments(core_name):
    url = "http://localhost:8983/solr/"+core_name + \
        "/update?commitWithin=1000&overwrite=true&wt=json"
    headers = {'Content-type': 'application/json'}
    data = {'delete': {'query': '*:*'}}
    requests.post(url=url, data=json.dumps(data), headers=headers)
    print("deleted old documents from "+core_name+" core")

In [13]:
def getData(core_name):
    # from 1 Month, there are about 90k entries. 2 clusters * 2 dcs * 8 instances * (4 weeks * 7 days * 24 hours * 4 (15min intervall))
    url = 'http://localhost:8983/solr/'+core_name+'/select?q=*:*&sort=timestamp%20asc&rows=100000'
    response = requests.get(url).json()['response']
    response
    return response['docs']

In [9]:
# get existing solr cores
#url = "http://localhost:8983/solr/admin/cores?action=STATUS"
#response = requests.get(url).json()
#activeCores = response['status'].keys()

# get existing elasticsearch indices
index_alias = "dc_cubes_*"
activeIndices = es.indices.get_alias(index_alias)
print("Active indices with alias: " + index_alias)
print(activeIndices)

In [208]:
if index_name in activeIndices:
    print(index_name + " already exists")
    # delete old data/predictions
    deleteElasticsearchIndex(index_name)
# else forecast core doesn't exist
else:
    print(index_name + " doesn't exist")
    # create an new forecast es index
    createElasticsearchIndex(index_name)
    # init schema
    #initSchema(index_name)

dc_cubes_historic already exists
deleted old documents from dc_cubes_historic core


In [207]:
# Read data from pickle file
with open("./data_pblm1.pkl", "rb") as pickleFile:
    pblm1 = pickle.load(pickleFile)
pblm1 = pblm1.reset_index()

In [29]:
# Read data from pickle file
with open("./data_pblm2.pkl", "rb") as pickleFile:
    pblm2 = pickle.load(pickleFile)
pblm2 = pblm2.reset_index()

In [210]:
pblm2.timestamp.max()

Timestamp('2020-01-23 09:30:00')

In [213]:
# missing 2020-01-23 09:30:00
copyRow = pblm2[pblm2.timestamp == pblm2.timestamp.max()].copy()

In [214]:
copyRow.timestamp = pd.Timestamp("2020-01-23 09:45:00")
copyRow

,index,timestamp,transactions_ps,opt_sga_size,sga_total,fixed_sga,java_pool,shared_pool,buffer_cache,pga_total,...,cursorcachehit_pct,bufcachehit_pct,redologalloc_hit_pct,cpuusage_pt,pxdwngrd25_ps,dictionarymiss_pct,cpuusage_ps,pxdwngrd50_ps,avg_tot_cpu_usage_ps,max_tot_cpu_usage_ps
2787,2787,2020-01-23 09:45:00,0.002,10240,20479.998,21.732,320,4928.0,14336,1213.082,...,106.049,100.0,100.0,268.818,0,0.0,0.449,0,3.005974,6.708903


In [216]:
pblm2 = pblm2.append(copyRow, ignore_index=True)

In [217]:
allMetrics = pblm2.columns

In [218]:
allMetrics = allMetrics.drop("timestamp")

In [219]:
pblm2[allMetrics] = pblm2[allMetrics].apply(pd.to_numeric, errors="coerce") # cast all to numeric because values arent getting pushed correctly to solr otherwise

In [220]:
pblm1[allMetrics] = pblm1[allMetrics].apply(pd.to_numeric, errors="coerce")

In [224]:
initSchema(core_name, allMetrics)

dc_cubes_historic  schema inited


In [227]:
# There are two dc's: 0 and 1
# dc 0 has clusters 6 and 8
# dc 1 has clusters 5 and 7
# Every Cluster has all 8 instances

def pushDataForAllInstances(df):
    instances = ["1","2","3","4","5","6","7","8" ]
    for instanz in instances:
        df["instanz"] = instanz
        df.apply(pushData, axis=1)

In [225]:
pblm2["dc"] = 0 # this will be pblm_2

In [226]:
pblm2["cluster"] = 6

In [228]:
pushDataForAllInstances(pblm2)

Commiting... counter: 90000
Commiting... counter: 91000
Commiting... counter: 92000
Commiting... counter: 93000
Commiting... counter: 94000
Commiting... counter: 95000
Commiting... counter: 96000
Commiting... counter: 97000
Commiting... counter: 98000
Commiting... counter: 99000
Commiting... counter: 100000
Commiting... counter: 101000
Commiting... counter: 102000
Commiting... counter: 103000
Commiting... counter: 104000
Commiting... counter: 105000
Commiting... counter: 106000
Commiting... counter: 107000
Commiting... counter: 108000
Commiting... counter: 109000
Commiting... counter: 110000
Commiting... counter: 111000


In [229]:
pblm2["cluster"] = 8

In [230]:
pushDataForAllInstances(pblm2)

Commiting... counter: 112000
Commiting... counter: 113000
Commiting... counter: 114000
Commiting... counter: 115000
Commiting... counter: 116000
Commiting... counter: 117000
Commiting... counter: 118000
Commiting... counter: 119000
Commiting... counter: 120000
Commiting... counter: 121000
Commiting... counter: 122000
Commiting... counter: 123000
Commiting... counter: 124000
Commiting... counter: 125000
Commiting... counter: 126000
Commiting... counter: 127000
Commiting... counter: 128000
Commiting... counter: 129000
Commiting... counter: 130000
Commiting... counter: 131000
Commiting... counter: 132000
Commiting... counter: 133000


In [231]:
pblm1["dc"] = 1

In [232]:
pblm1["cluster"] = 5

In [233]:
pushDataForAllInstances(pblm1)

Commiting... counter: 134000
Commiting... counter: 135000
Commiting... counter: 136000
Commiting... counter: 137000
Commiting... counter: 138000
Commiting... counter: 139000
Commiting... counter: 140000
Commiting... counter: 141000
Commiting... counter: 142000
Commiting... counter: 143000
Commiting... counter: 144000
Commiting... counter: 145000
Commiting... counter: 146000
Commiting... counter: 147000
Commiting... counter: 148000
Commiting... counter: 149000
Commiting... counter: 150000
Commiting... counter: 151000
Commiting... counter: 152000
Commiting... counter: 153000
Commiting... counter: 154000
Commiting... counter: 155000
Commiting... counter: 156000


In [234]:
pblm1["cluster"] = 7

In [235]:
pushDataForAllInstances(pblm1)

Commiting... counter: 157000
Commiting... counter: 158000
Commiting... counter: 159000
Commiting... counter: 160000
Commiting... counter: 161000
Commiting... counter: 162000
Commiting... counter: 163000
Commiting... counter: 164000
Commiting... counter: 165000
Commiting... counter: 166000
Commiting... counter: 167000
Commiting... counter: 168000
Commiting... counter: 169000
Commiting... counter: 170000
Commiting... counter: 171000
Commiting... counter: 172000
Commiting... counter: 173000
Commiting... counter: 174000
Commiting... counter: 175000
Commiting... counter: 176000
Commiting... counter: 177000
Commiting... counter: 178000


In [236]:
requests.get("http://localhost:8983/solr/"+core_name+"/update?commit=true")

<Response [200]>

In [237]:
#*______________________________________*#

In [1]:
forecast_core_name = "dc_cubes_forecast"

In [2]:
historicCoreName = "dc_cubes_historic"

In [14]:
hist_df = pd.DataFrame.from_dict(getData(historicCoreName))

In [15]:
allColumns = hist_df.columns.to_list()

In [16]:
url = "http://localhost:8983/solr/admin/cores?action=STATUS"
response = requests.get(url).json()
activeCores = response['status'].keys()

# if forecast core exists
if forecast_core_name in activeCores:
    print(forecast_core_name + " already exists")
    deleteSolrCore(forecast_core_name)
    # delete old data/predictions
    createSolrCore(forecast_core_name)
    # init schema
    initSchema(forecast_core_name, allColumns)
    #deleteCoreDocuments(forecast_core_name)
# else forecast core doesn't exist
else:
    print(forecast_core_name + " doesn't exist")
    # create an new forecast solr core
    createSolrCore(forecast_core_name)
    # init schema
    initSchema(forecast_core_name, allColumns)

dc_cubes_forecast already exists
dc_cubes_forecast  core deleted
dc_cubes_forecast  created
dc_cubes_forecast  schema inited


In [17]:
# splits the data of each cube from the whole df in its own dataframe and takes only the last -history_steps
def splitInCubesFrames(df):
    unique_server_names = df.server.unique()
    splitted_frames = []
    for name in unique_server_names:
        new_df = df[df['server'] == name].copy()
        splitted_frames.append(new_df)
    return splitted_frames


In [18]:

measureInterval = 15 #min
hoursToPredict = 24 * 4
pred_horizon = int((60//measureInterval) * hoursToPredict) #(4*hours), timestep = 15min
days_history = 1
hours_history = int(24 * days_history)
n_history = int((60//measureInterval)*hours_history)

history_steps = n_history
forecast_steps = pred_horizon

In [19]:
timestamp = "timestamp"

In [20]:
hist_df.reset_index(inplace=True)

In [21]:
last_timestamp = hist_df.timestamp.max()
hist_df[timestamp] = pd.to_datetime(hist_df.timestamp)
# generate features/columns from the timestamp
hist_df["dayOfWeek"] = hist_df[timestamp].map(lambda x: x.dayofweek)
hist_df["isWeekend"] = hist_df.dayOfWeek.map(lambda x: 0 if (x < 5) else 1) # saturday.dayofweek = 5, monday=0
#hist_df["weekofyear"] = hist_df[timestamp].map(lambda x: x.weekofyear)
hist_df["hour"] = hist_df[timestamp].map(lambda x: x.hour)
hist_df["minute"]= hist_df[timestamp].map(lambda x: x.minute)
hist_df = hist_df.set_index('timestamp')

hist_df.index = pd.to_datetime(hist_df.index).sort_values()

In [22]:
from keras.models import load_model

# split cubes in own frames
cubes_frames = splitInCubesFrames(hist_df)

# load the trained model
modelDc0 = load_model('final_cnn_pblm2.h5')
modelDc1 = load_model('final_cnn_pblm1.h5')

Using TensorFlow backend.


In [41]:
def makePredictionFrame(modelDc0, modelDc1, cubes_frames, last_timestamp, predictionColumn = "cpuusage_ps"):
    prediction_frames = []
    for cube in cubes_frames:
        # transform pre prediction input

        # extracting information from the current server
        last_timestamp = cube.index[-1]
        server_name = cube['server'].iloc[0]
        cluster = cube['cluster'].iloc[0]
        dc = cube['dc'].iloc[0]
        perm = cube['perm'].iloc[0]
        instanz = cube['instanz'].iloc[0]
        verfahren = cube['verfahren'].iloc[0]
        service = cube['service'].iloc[0]

        print("DATACENTER: ", dc)
        if(int(dc) == 0):
            model = modelDc0
        else:
            model = modelDc1
            
        dropCols = ["id", "cluster",
                "_version_",
                "dc",
                "perm",
                "instanz",
                "verfahren",
                "service",
                "response",
                "minv",
                "maxv", 
                "avg",
                "var",
                "dev_upp",
                "dev_low",
                "perc90",
                "perc95",
                "perc99",
                "sum",
                "sum_of_squares",
                "server"]
        # Converting the index as date
#         pdb.set_trace()
        cube.index = pd.to_datetime(cube.index).sort_values()
        


        cube = cube.drop(dropCols,axis=1)
        dataset = cube
#         if dc == 0:
#             allCols = cube.columns
#             cube[allCols] = cube[allCols].apply(pd.to_numeric, errors="coerce")
#             dataset = cube.dropna(axis=1, how="all")
#             dataset["dummy1"] = 1
#             dataset["dummy2"] = 0
        y = dataset[predictionColumn].copy() # prediction column was pushed to count
        x = dataset.drop(columns=[predictionColumn, "count"])
#         print(x.shape)
        # Standardise
        allMetrics =  dataset.columns.tolist()
        print("X SHAPE", x.shape)

        # standardise
        scalerX = StandardScaler()
        scalerX.fit(x)
        x = scalerX.transform(x)
        scalerY = StandardScaler()
       # .reshape(-1, 1) # needed for standardScaler
        scalerY.fit(y.values.reshape(-1,1))
        y = scalerY.transform(y.values.reshape(-1,1))
        
        #PCA
#         pcaTransformer = PCA(n_components=63) # keep 95% variance
#         pcaTransformer.fit(x)
#         x = pcaTransformer.transform(x)

        #transformed_df = pd.DataFrame().from_records(x)
        #transformed_df[predictionColumn] = y
        numberOfFeatures = x.shape[1]
        # predict
        pred_input = x[x.shape[0]-history_steps:]
#         print("***mean, std", x.mean(), x.std())
#         print("************\n", pred_input)
#         pdb.set_trace()
        pred_input = pred_input.reshape(
            (1, history_steps, numberOfFeatures)) # alternative (1, pred_input.shape[0],pred_input.shape[1])
        prediction = model.predict(pred_input)
#         print("prediction: ", prediction, prediction.shape)
        #prediction = np.hstack((prediction, np.zeros((prediction.shape[0], numberOfFeatures-1), dtype=prediction.dtype)))
        
    #    print("pre inverse scaling", prediction)
        prediction = scalerY.inverse_transform(prediction)
#        print("post inverse scaling", prediction)
        # transform pre solr
#         prediction = prediction.reshape((forecast_steps, 1))
#         prediction = np.hstack((prediction, np.zeros(
#             (prediction.shape[0], 3), dtype=prediction.dtype)))
#         prediction = prediction = scaler.inverse_transform(prediction)
#         prediction = prediction[:, [0]]
#         int_prediction = prediction.astype(int, copy=True)
#         prediction = int_prediction

        prediction = prediction.reshape(prediction.shape[1])
        # make the dataframe
        next_timestamps = pd.date_range(
            start=last_timestamp, periods=forecast_steps+1, freq='15min',  closed='right')
        # create the prediction dataframe for the current server
        d = {'timestamp': next_timestamps, 'cluster': cluster, 'dc': dc,
             'perm': perm, 'instanz': instanz,  'verfahren': verfahren, 'service': service, 'response': 200}
        pred_df = pd.DataFrame(data=d)
        for metric in allMetrics:
            pred_df[metric] = -1
#         print("Length:", len(pred_df), "::::::", prediction.shape)
        pred_df['count'] = prediction
        pred_df[predictionColumn] = prediction
        pred_df['minv'] = 0
        pred_df['maxv'] = 0
        pred_df['avg'] = 0
        pred_df['var'] = 0
        pred_df['dev_upp'] = 0
        pred_df['dev_low'] = 0
        pred_df['perc90'] = 0
        pred_df['perc95'] = 0
        pred_df['perc99.9'] = 0
        pred_df['sum'] = 0
        pred_df['sum_of_squares'] = 0
        pred_df['server'] = server_name

        pred_df['timestamp'] = pred_df['timestamp'].dt.strftime(
            '%Y-%m-%dT%H:%M:00Z')
        prediction_frames.append(pred_df)
    print("Made predictions")
    return pd.concat(prediction_frames, ignore_index=True)



In [42]:
prediction_df = makePredictionFrame(modelDc0, modelDc1, cubes_frames, last_timestamp)

DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  0
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHAPE (2789, 176)
DATACENTER:  1
X SHA

In [43]:
def pushForecastData(row):
    global forecast_core_name
    global counter
    global allColumns
    # defining the api-endpoint
    url = "http://localhost:8983/solr/"+forecast_core_name+"/update/json/docs"
    # data to be sent to api
    data = {
                "timestamp": row["timestamp"],
                "cluster": row["cluster"],
                "dc": row["dc"],
                "perm": row["perm"],
                "instanz": row["instanz"],
                "verfahren": row["verfahren"],
                "service": row["verfahren"],
                "response": row["response"],
                "count": row["count"],
                "minv": row["minv"],
                "maxv": row["maxv"],
                "avg": row["avg"],
                "var": row["var"],
                "dev_upp": row["dev_upp"],
                "dev_low": row["dev_low"],
                "perc90": row["perc90"],
                "perc95": row["perc95"],
                "perc99": row["perc99.9"],
                "sum": row["sum"],
                "sum_of_squares": row["sum_of_squares"],
                "server": row["server"],
                "cpuusage_ps": row["cpuusage_ps"]
            }
    
    for col in allColumns:
        if col not in data:
            data[col] = -1
    data.pop("id", None)
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.get(url)
    headers = {'Content-type': 'application/json'}
    # sending post request
    session.post(url=url, data=json.dumps(data), headers=headers)
    counter += 1
    if (counter % 1000 == 0):
        print("Commiting... counter:", counter)
        requests.get("http://localhost:8983/solr/"+forecast_core_name+"/update?commit=true")

In [44]:
prediction_df.head()

,timestamp,cluster,dc,perm,instanz,verfahren,service,response,level_0,count,...,avg,var,dev_upp,dev_low,perc90,perc95,perc99.9,sum,sum_of_squares,server
0,2020-01-23T10:00:00Z,8,0,-1,1,-1,-1,200,-1,1.439913,...,0,0,0,0,0,0,0,0,0,PBZ08E00_PERM02_S01_OSB
1,2020-01-23T10:15:00Z,8,0,-1,1,-1,-1,200,-1,1.080803,...,0,0,0,0,0,0,0,0,0,PBZ08E00_PERM02_S01_OSB
2,2020-01-23T10:30:00Z,8,0,-1,1,-1,-1,200,-1,0.854486,...,0,0,0,0,0,0,0,0,0,PBZ08E00_PERM02_S01_OSB
3,2020-01-23T10:45:00Z,8,0,-1,1,-1,-1,200,-1,0.865069,...,0,0,0,0,0,0,0,0,0,PBZ08E00_PERM02_S01_OSB
4,2020-01-23T11:00:00Z,8,0,-1,1,-1,-1,200,-1,1.048856,...,0,0,0,0,0,0,0,0,0,PBZ08E00_PERM02_S01_OSB


In [45]:
prediction_df.apply(pushForecastData, axis=1)
print("Last Commit...")
requests.get("http://localhost:8983/solr/"+forecast_core_name+"/update?commit=true")

Commiting... counter: 1000
Commiting... counter: 2000
Commiting... counter: 3000
Commiting... counter: 4000
Commiting... counter: 5000
Commiting... counter: 6000
Commiting... counter: 7000
Commiting... counter: 8000
Commiting... counter: 9000
Commiting... counter: 10000
Commiting... counter: 11000
Commiting... counter: 12000
Last Commit...


<Response [200]>

In [46]:
merged_core_name = "dc_cubes_merged"

In [47]:
def mergeTwoCores(merged_core_name, core1, core2):
    url = "http://localhost:8983/solr/admin/cores?action=mergeindexes&core=" + merged_core_name + "&srcCore=" + core1 + "&srcCore=" + core2
    requests.get(url=url)
    # Commit to materialize changes
    requests.get("http://localhost:8983/solr/"+merged_core_name+"/update?commit=true")
    print(core1 + " and " +  core2 + " have been merged to " + merged_core_name)

In [49]:
# if merged core exists
if merged_core_name in activeCores:
    print(merged_core_name + " already exists")
    # delete old data/predictions
    deleteCoreDocuments(merged_core_name)
# else forecast core doesn't exist
else:
    print(merged_core_name + " doesn't exist")
    # create an new forecast solr core
    createSolrCore(merged_core_name)
    # init schema
    initSchema(merged_core_name, allColumns)

# merged historic and forecast core
mergeTwoCores(merged_core_name, historicCoreName, forecast_core_name)

dc_cubes_merged already exists
deleted old documents from dc_cubes_merged core
dc_cubes_historic and dc_cubes_forecast have been merged to dc_cubes_merged
